# Healthcare Fraud Detection via Knowledge Graphs
## 🔐 INSTRUCTOR SOLUTION — DO NOT DISTRIBUTE

This notebook detects all five fraud patterns embedded in the 2024 synthetic claims dataset.

| # | Pattern | Fraudster | Graph Signal |
|---|---------|-----------|---------------|
| 1 | **Ghost Billing** | PRV-047 | Temporal edge after `date_of_death` |
| 2 | **Referral Ring** | PRV-031/032/033 | Closed Louvain community, inflated amounts |
| 3 | **Impossible Travel** | PAT-0089 | Same patient, same day, two distant cities |
| 4 | **Upcoding** | PRV-022 | 99215 billing rate 8× above peers (3σ outlier) |
| 5 | **Duplicate Billing** | PRV-055 | Parallel edges in multigraph |

## 1  Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import community as community_louvain
from pyvis.network import Network
from collections import Counter

pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-darkgrid')

COLORS = {
    'GHOST_BILLING':     '#e74c3c',
    'REFERRAL_RING':     '#e67e22',
    'IMPOSSIBLE_TRAVEL': '#9b59b6',
    'UPCODING':          '#3498db',
    'DUPLICATE_BILLING': '#27ae60',
    'normal':            '#bdc3c7',
}
print('Imports OK')

## 2  Load Data

In [ ]:
patients  = pd.read_csv('../data/patients.csv',  parse_dates=['date_of_birth', 'date_of_death'])
providers = pd.read_csv('../data/providers.csv')
claims    = pd.read_csv('../data/claims.csv',    parse_dates=['date_of_service'])
referrals = pd.read_csv('../data/referrals.csv', parse_dates=['referral_date'])

fraud = claims[claims['is_fraud']]
print(f'Patients:  {len(patients):,}')
print(f'Providers: {len(providers):,}')
print(f'Claims:    {len(claims):,}')
print(f'Referrals: {len(referrals):,}')
print(f'\nFraud claims:  {len(fraud):,}  ({len(fraud)/len(claims)*100:.1f}%)')
print(f'Fraud billed:  ${fraud["amount_billed"].sum():,.2f}')
print()
print(fraud.groupby('fraud_type')[['claim_id','amount_billed']]
      .agg(n_claims=('claim_id','count'), total_billed=('amount_billed','sum')))

## 3  Build the Knowledge Graph

We represent the dataset as a **heterogeneous multigraph**:

- **Patient nodes** (`PAT-*`) — with city, deceased flag
- **Provider nodes** (`PRV-*`) — with city, specialty, name
- **Claim edges** — one edge per claim, connecting patient to provider

A separate **directed referral graph** links providers to each other.

In [ ]:
G = nx.MultiGraph()

for _, row in patients.iterrows():
    G.add_node(row['patient_id'],
               ntype='patient',
               city=row['city'],
               deceased=pd.notna(row['date_of_death']))

for _, row in providers.iterrows():
    G.add_node(row['provider_id'],
               ntype='provider',
               city=row['city'],
               name=row['provider_name'])

for _, row in claims.iterrows():
    G.add_edge(row['patient_id'], row['provider_id'],
               claim_id=row['claim_id'],
               amount=row['amount_billed'],
               date=row['date_of_service'],
               procedure=row['procedure_code'],
               is_fraud=row['is_fraud'])

print(f'Nodes: {G.number_of_nodes():,}  (patients + providers)')
print(f'Edges: {G.number_of_edges():,}  (claims)')

prov_nodes = [n for n, d in G.nodes(data=True) if d.get('ntype') == 'provider']
degrees = sorted([G.degree(n) for n in prov_nodes], reverse=True)
print(f'\nProvider degree — max: {degrees[0]}  median: {np.median(degrees):.0f}  min: {degrees[-1]}')

---
## 4  Fraud Pattern 1 — Ghost Billing

**What it is:** A provider bills for services allegedly rendered to patients who are already deceased.

**Graph signal:** A temporal edge whose `date` attribute falls *after* the patient node’s `date_of_death`.

In [ ]:
deceased = patients[patients['date_of_death'].notna()].copy()

ghost = claims.merge(deceased[['patient_id', 'date_of_death']], on='patient_id')
ghost = ghost[ghost['date_of_service'] > ghost['date_of_death']]

print(f'Claims filed AFTER patient death: {len(ghost)}')
print(f'  Provider(s):  {ghost["provider_id"].unique()}')
print(f'  Patients:     {ghost["patient_id"].unique()}')
print(f'  Amount:       ${ghost["amount_billed"].sum():,.2f}')
print()
print(ghost[['claim_id','patient_id','provider_id','date_of_service',
             'date_of_death','amount_billed']].head(8).to_string(index=False))

In [ ]:
ghost_pids = list(ghost['patient_id'].unique())
ghost_prvs = list(ghost['provider_id'].unique())
sub = G.subgraph(ghost_pids + ghost_prvs)

pos = nx.bipartite_layout(sub, ghost_prvs)
fig, ax = plt.subplots(figsize=(8, 4))

nx.draw_networkx_nodes(sub, pos, nodelist=ghost_prvs,
                       node_color=COLORS['GHOST_BILLING'], node_size=900, ax=ax)
nx.draw_networkx_nodes(sub, pos, nodelist=ghost_pids,
                       node_color='#555', node_size=400, ax=ax)
nx.draw_networkx_labels(sub, pos, ax=ax, font_size=7, font_color='white')
nx.draw_networkx_edges(sub, pos, ax=ax,
                       edge_color=COLORS['GHOST_BILLING'], width=1.5, alpha=0.7)

ax.set_title('Ghost Billing: PRV-047 billing deceased patients (grey)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('../data/fraud1_ghost_billing.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5  Fraud Pattern 2 — Referral Ring

**What it is:** Three providers refer patients exclusively to each other (zero external referrals) and inflate claim amounts 2–3×.

**Graph signal:** A small, *closed* community in the provider-referral graph. Louvain detects it, and the community’s external-edge count is zero. Betweenness centrality of ring members is near zero — they sit in a dead-end pocket.

In [ ]:
R = nx.DiGraph()
for _, row in referrals.iterrows():
    u, v = row['referring_provider_id'], row['referred_provider_id']
    if R.has_edge(u, v):
        R[u][v]['weight'] += 1
    else:
        R.add_edge(u, v, weight=1)

print(f'Referral graph: {R.number_of_nodes()} nodes, {R.number_of_edges()} edges')

R_und     = R.to_undirected()
partition = community_louvain.best_partition(R_und, random_state=42)
comm_sizes = Counter(partition.values())
print(f'Communities found: {len(comm_sizes)}')
print(f'Size distribution (top 10): {sorted(comm_sizes.values(), reverse=True)[:10]}')

In [ ]:
closed_communities = []
for cid, size in comm_sizes.items():
    if not (2 <= size <= 6):
        continue
    members = {n for n, c in partition.items() if c == cid}
    ext = [(u, v) for u, v in R.edges() if (u in members) != (v in members)]
    if len(ext) == 0:
        closed_communities.append(members)
        ring_claims = claims[claims['provider_id'].isin(members)]
        avg_ring  = ring_claims['amount_billed'].mean()
        avg_all   = claims['amount_billed'].mean()
        print(f'CLOSED COMMUNITY: {sorted(members)}')
        print(f'  Claims:       {len(ring_claims)}')
        print(f'  Total billed: ${ring_claims["amount_billed"].sum():,.2f}')
        print(f'  Avg claim:    ${avg_ring:.2f}  vs overall ${avg_all:.2f}  ({avg_ring/avg_all:.1f}x multiplier)')

In [ ]:
ring_members = {n for members in closed_communities for n in members}
pos = nx.spring_layout(R_und, seed=42, k=1.5)
n_comm = max(partition.values()) + 1
cmap   = plt.cm.tab20

node_colors, node_sizes = [], []
for n in R_und.nodes():
    if n in ring_members:
        node_colors.append(COLORS['REFERRAL_RING']); node_sizes.append(500)
    else:
        node_colors.append(cmap(partition[n] / n_comm)); node_sizes.append(150)

fig, ax = plt.subplots(figsize=(11, 7))
nx.draw_networkx_edges(R_und, pos, ax=ax, alpha=0.15, edge_color='grey', width=0.5)
nx.draw_networkx_nodes(R_und, pos, ax=ax,
                       node_color=node_colors, node_size=node_sizes, alpha=0.9)
nx.draw_networkx_labels(R_und, pos, ax=ax,
                        labels={n: n for n in ring_members},
                        font_size=7, font_color='white')
ax.set_title('Provider Referral Network — Louvain Communities\n'
             'Orange = closed fraud ring (PRV-031/032/033)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig('../data/fraud2_referral_ring.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6  Fraud Pattern 3 — Impossible Travel

**What it is:** A patient has claims at two geographically distant providers on the *same calendar day* — physically impossible.

**Graph signal:** In a patient–(provider, location, date) tripartite view, a patient node has two edges with the same date but conflicting city attributes.

In [ ]:
c = claims.merge(providers[['provider_id', 'city']], on='provider_id')

multi_city = (c.groupby(['patient_id', 'date_of_service'])['city']
               .nunique().reset_index(name='n_cities'))
impossible = multi_city[multi_city['n_cities'] > 1]

print(f'Patient-days with claims in >1 city: {len(impossible)}')
print(f'Affected patients: {impossible["patient_id"].unique()}')
print()

suspect = impossible['patient_id'].iloc[0]
detail  = (c[c['patient_id'] == suspect]
           [['claim_id','patient_id','provider_id','city','date_of_service','amount_billed']]
           .sort_values('date_of_service'))
# Show only the flagged days
flagged_dates = impossible[impossible['patient_id'] == suspect]['date_of_service']
print(detail[detail['date_of_service'].isin(flagged_dates)].to_string(index=False))

In [ ]:
pat_claims = c[c['patient_id'] == suspect].copy()
city_colors = {'New York': '#3498db', 'Los Angeles': '#e74c3c'}

fig, ax = plt.subplots(figsize=(13, 3))
for city in pat_claims['city'].unique():
    sub = pat_claims[pat_claims['city'] == city]
    ax.scatter(sub['date_of_service'], [city]*len(sub),
               c=city_colors.get(city, '#7f8c8d'), s=120, zorder=5, label=city)

for dt in flagged_dates:
    ax.axvline(dt, color=COLORS['IMPOSSIBLE_TRAVEL'], alpha=0.4, linewidth=10, zorder=1)

ax.set_title(f'PAT-0089: same-day claims in two cities (purple bars = impossible-travel days)', fontsize=11)
ax.set_xlabel('Date of Service')
ax.legend(title='Provider City', loc='upper right')
plt.tight_layout()
plt.savefig('../data/fraud3_impossible_travel.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7  Fraud Pattern 4 — Upcoding

**What it is:** A provider systematically bills code 99215 (highest-complexity office visit, ~$320) for ~88% of visits. The population average is ~12%.

**Graph signal:** In a bipartite provider–procedure_code graph, the edge-weight distribution of one provider is a 3σ outlier.

In [ ]:
office_codes  = ['99213', '99214', '99215', '99203', '99204']
office_claims = claims[claims['procedure_code'].isin(office_codes)]

dist = (office_claims.groupby(['provider_id', 'procedure_code'])
                     .size().reset_index(name='n'))
dist['pct'] = dist.groupby('provider_id')['n'].transform(lambda x: x / x.sum())

c99215   = dist[dist['procedure_code'] == '99215'].copy()
pop_mean = c99215['pct'].mean()
pop_std  = c99215['pct'].std()

print(f'Population 99215 rate: mean={pop_mean:.1%}  std={pop_std:.1%}')
print()

outliers = c99215[c99215['pct'] > pop_mean + 3 * pop_std].copy()
outliers['z_score'] = (outliers['pct'] - pop_mean) / pop_std
print('Providers > 3σ above mean:')
print(outliers[['provider_id', 'pct', 'z_score']].sort_values('pct', ascending=False).to_string(index=False))

In [ ]:
plot_data   = c99215.sort_values('pct', ascending=False)
bar_colors  = [COLORS['UPCODING'] if r == 'PRV-022' else COLORS['normal']
               for r in plot_data['provider_id']]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(plot_data)), plot_data['pct'], color=bar_colors, width=0.8)
ax.axhline(pop_mean, color='white', linestyle='--', linewidth=1.2,
           label=f'Mean ({pop_mean:.1%})')
ax.axhline(pop_mean + 3*pop_std, color='#e74c3c', linestyle=':',  linewidth=1.2,
           label='3σ threshold')

for i, (_, row) in enumerate(plot_data.iterrows()):
    if row['provider_id'] == 'PRV-022':
        ax.text(i, row['pct'] + 0.01, 'PRV-022', ha='center',
                fontsize=8, color=COLORS['UPCODING'], fontweight='bold')

ax.set_xlabel('Providers (sorted by 99215 rate, highest first)')
ax.set_ylabel('Fraction of office visits billed as 99215')
ax.set_title('Upcoding Detection: 99215 Billing Rate per Provider', fontsize=11)
ax.set_xticks([])
ax.legend()
plt.tight_layout()
plt.savefig('../data/fraud4_upcoding.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8  Fraud Pattern 5 — Duplicate Billing

**What it is:** The provider re-submits the same claim 1–2 extra times with slight amount jitter to avoid exact-match detection.

**Graph signal:** In the MultiGraph, *parallel edges* between the same (patient, provider) pair share the same `date` and `procedure` attributes.

In [ ]:
key = ['patient_id', 'provider_id', 'procedure_code', 'date_of_service']
dup_groups = (claims.groupby(key)
                    .agg(n_copies=('claim_id','count'),
                         total_billed=('amount_billed','sum'),
                         claim_ids=('claim_id', list))
                    .reset_index()
                    .query('n_copies > 1')
                    .sort_values('n_copies', ascending=False))

print(f'Duplicate claim groups: {len(dup_groups)}')
all_dup_ids = [c for grp in dup_groups['claim_ids'] for c in grp]
print(f'Providers involved: {claims.loc[claims["claim_id"].isin(all_dup_ids), "provider_id"].unique()}')

extra = (dup_groups['total_billed'] * (1 - 1/dup_groups['n_copies'])).sum()
print(f'Fraudulent extra payments: ${extra:,.2f}')
print()
print(dup_groups[['patient_id','provider_id','procedure_code',
                  'date_of_service','n_copies','total_billed']].head(8).to_string(index=False))

In [ ]:
# Multigraph parallel-edge view for PRV-055
prv055_edges = list(G.edges('PRV-055', data=True))
edge_counts  = {}
for u, v, d in prv055_edges:
    k = (u, v, d['procedure'], str(d['date'])[:10])
    edge_counts[k] = edge_counts.get(k, 0) + 1

dupes = {k: v for k, v in edge_counts.items() if v > 1}
print(f'Duplicate edge groups for PRV-055 in the MultiGraph: {len(dupes)}')
for k, cnt in list(dupes.items())[:6]:
    print(f'  Patient {k[0]}  proc {k[2]}  date {k[3]}  x{cnt}')

---
## 9  Interactive Knowledge Graph

Provider-centric view of the referral network. Open the saved HTML in a browser for interactive exploration.
Fraud providers are highlighted in their pattern colour.

In [ ]:
net = Network(height='700px', width='100%', bgcolor='#1a1a2e',
              font_color='white', directed=True,
              notebook=True, cdn_resources='in_line')

FRAUD_TAG = {
    'PRV-047': ('GHOST_BILLING',     'Ghost'),
    'PRV-031': ('REFERRAL_RING',     'Ring'),
    'PRV-032': ('REFERRAL_RING',     'Ring'),
    'PRV-033': ('REFERRAL_RING',     'Ring'),
    'PRV-022': ('UPCODING',          'Upcode'),
    'PRV-055': ('DUPLICATE_BILLING', 'Dupe'),
}

for _, prov in providers.iterrows():
    pid   = prov['provider_id']
    info  = FRAUD_TAG.get(pid)
    color = COLORS[info[0]] if info else COLORS['normal']
    label = f"{pid}\n{info[1]}" if info else pid
    size  = 40 if info else 20
    title = f"<b>{prov['provider_name']}</b><br>{prov['specialty']}<br>{prov['city']}, {prov['state']}"
    net.add_node(pid, label=label, color=color, size=size, title=title)

for _, row in referrals.iterrows():
    net.add_edge(row['referring_provider_id'], row['referred_provider_id'],
                 color='rgba(200,200,200,0.2)', width=1)

out_path = '../data/provider_network.html'
net.save_graph(out_path)
print(f'Saved interactive graph to {out_path}')
net.show(out_path)

---
## 10  Summary

In [ ]:
rows = [
    ('Ghost Billing',    'PRV-047',              len(ghost),
     ghost['amount_billed'].sum(),
     'Temporal: service date after patient death'),
    ('Referral Ring',    'PRV-031/032/033',
     len(claims[claims['fraud_type']=='REFERRAL_RING']),
     claims[claims['fraud_type']=='REFERRAL_RING']['amount_billed'].sum(),
     'Community: closed Louvain cluster, 2-3x amounts'),
    ('Impossible Travel','PAT-0089',             len(impossible)*2,
     claims[claims['fraud_type']=='IMPOSSIBLE_TRAVEL']['amount_billed'].sum(),
     'Bipartite temporal: 2 cities on same day'),
    ('Upcoding',         'PRV-022',
     len(claims[claims['fraud_type']=='UPCODING']),
     claims[claims['fraud_type']=='UPCODING']['amount_billed'].sum(),
     'Degree outlier: 99215 rate 8x above mean (3σ)'),
    ('Duplicate Billing','PRV-055',
     len(claims[claims['fraud_type']=='DUPLICATE_BILLING']),
     claims[claims['fraud_type']=='DUPLICATE_BILLING']['amount_billed'].sum(),
     'Multigraph: parallel edges (same pat/proc/date)'),
]

summary = pd.DataFrame(rows, columns=['Pattern','Fraudster','Claims','Fraud USD','Detection Method'])
summary['Fraud USD'] = summary['Fraud USD'].map('${:,.2f}'.format)
print(summary.to_string(index=False))
print(f'\nTotal fraud: ${claims[claims["is_fraud"]]["amount_billed"].sum():,.2f}')
print(f'As % total:  {claims[claims["is_fraud"]]["amount_billed"].sum()/claims["amount_billed"].sum()*100:.1f}%')